In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

In [3]:
data_path = Path("../data/hf_external_validation_data/external_validation_data.csv")

df = pd.read_csv(data_path)

print("Dataset Loaded Successfully")

print("Shape :", df.shape)

df.head()

Dataset Loaded Successfully
Shape : (1552210, 44)


,Unnamed: 0,Hour,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,FiO2,pH,PaCO2,SaO2,AST,BUN,Alkalinephos,Calcium,Chloride,Creatinine,Bilirubin_direct,Glucose,Lactate,Magnesium,Phosphate,Potassium,Bilirubin_total,TroponinI,Hct,Hgb,PTT,WBC,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,Patient_ID
0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,68.54,0,NaN,NaN,-0.02,1,0,17072
1,1,1,65.0,100.0,NaN,NaN,72.0,NaN,16.5,NaN,NaN,NaN,0.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,68.54,0,NaN,NaN,-0.02,2,0,17072
2,2,2,78.0,100.0,NaN,NaN,42.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,68.54,0,NaN,NaN,-0.02,3,0,17072
3,3,3,73.0,100.0,NaN,NaN,NaN,NaN,17.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,68.54,0,NaN,NaN,-0.02,4,0,17072
4,4,4,70.0,100.0,NaN,129.0,74.0,69.0,14.0,NaN,NaN,26.0,0.4,NaN,NaN,NaN,NaN,23.0,NaN,9.6,104.0,0.8,NaN,161.0,NaN,1.6,2.1,3.2,NaN,NaN,29.7,9.5,30.6,11.3,NaN,330.0,68.54,0,NaN,NaN,-0.02,5,0,17072


In [4]:
print("Columns:\n")
print(df.columns.tolist())

print("\nUnique Patients :", df["Patient_ID"].nunique())

print("\nRows :", len(df))

print("\nSepsis Distribution:")
print(df["SepsisLabel"].value_counts())

Columns:

['Unnamed: 0', 'Hour', 'HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2', 'BaseExcess', 'HCO3', 'FiO2', 'pH', 'PaCO2', 'SaO2', 'AST', 'BUN', 'Alkalinephos', 'Calcium', 'Chloride', 'Creatinine', 'Bilirubin_direct', 'Glucose', 'Lactate', 'Magnesium', 'Phosphate', 'Potassium', 'Bilirubin_total', 'TroponinI', 'Hct', 'Hgb', 'PTT', 'WBC', 'Fibrinogen', 'Platelets', 'Age', 'Gender', 'Unit1', 'Unit2', 'HospAdmTime', 'ICULOS', 'SepsisLabel', 'Patient_ID']

Unique Patients : 40336

Rows : 1552210

Sepsis Distribution:
SepsisLabel
0    1524294
1      27916
Name: count, dtype: int64


In [8]:
exclude_columns = [
    "Unnamed: 0",
    "Patient_ID",
    "SepsisLabel",
    "Hour"
]

numeric_columns = df.select_dtypes(include=np.number).columns #in this only the numeric columns are picked ignoring the text related data

numeric_columns = [
    col for col in numeric_columns
    if col not in exclude_columns
]

print("Number of Physiological Features :", len(numeric_columns))
print(numeric_columns)

Number of Physiological Features : 40
['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2', 'BaseExcess', 'HCO3', 'FiO2', 'pH', 'PaCO2', 'SaO2', 'AST', 'BUN', 'Alkalinephos', 'Calcium', 'Chloride', 'Creatinine', 'Bilirubin_direct', 'Glucose', 'Lactate', 'Magnesium', 'Phosphate', 'Potassium', 'Bilirubin_total', 'TroponinI', 'Hct', 'Hgb', 'PTT', 'WBC', 'Fibrinogen', 'Platelets', 'Age', 'Gender', 'Unit1', 'Unit2', 'HospAdmTime', 'ICULOS']


In [9]:
external_feature_dataset = []

for patient_id, patient in df.groupby("Patient_ID"):

    features = {}

    for column in numeric_columns:

        features[column + "_mean"] = patient[column].mean()
        features[column + "_min"] = patient[column].min()
        features[column + "_max"] = patient[column].max()
        features[column + "_std"] = patient[column].std()
        features[column + "_last"] = patient[column].iloc[-1]

    # Patient label
    features["SepsisLabel"] = patient["SepsisLabel"].max()

    external_feature_dataset.append(features)

In [10]:
### converstion to dataframe
external_feature_dataset = pd.DataFrame(external_feature_dataset)

print("Shape :", external_feature_dataset.shape)

external_feature_dataset.head()

Shape : (40336, 201)


,HR_mean,HR_min,HR_max,HR_std,HR_last,O2Sat_mean,O2Sat_min,O2Sat_max,O2Sat_std,O2Sat_last,Temp_mean,Temp_min,Temp_max,Temp_std,Temp_last,SBP_mean,SBP_min,SBP_max,SBP_std,SBP_last,MAP_mean,MAP_min,MAP_max,MAP_std,MAP_last,DBP_mean,DBP_min,DBP_max,DBP_std,DBP_last,Resp_mean,Resp_min,Resp_max,Resp_std,Resp_last,EtCO2_mean,EtCO2_min,EtCO2_max,EtCO2_std,EtCO2_last,BaseExcess_mean,BaseExcess_min,BaseExcess_max,BaseExcess_std,BaseExcess_last,HCO3_mean,HCO3_min,HCO3_max,HCO3_std,HCO3_last,FiO2_mean,FiO2_min,FiO2_max,FiO2_std,FiO2_last,pH_mean,pH_min,pH_max,pH_std,pH_last,PaCO2_mean,PaCO2_min,PaCO2_max,PaCO2_std,PaCO2_last,SaO2_mean,SaO2_min,SaO2_max,SaO2_std,SaO2_last,AST_mean,AST_min,AST_max,AST_std,AST_last,BUN_mean,BUN_min,BUN_max,BUN_std,BUN_last,Alkalinephos_mean,Alkalinephos_min,Alkalinephos_max,Alkalinephos_std,Alkalinephos_last,Calcium_mean,Calcium_min,Calcium_max,Calcium_std,Calcium_last,Chloride_mean,Chloride_min,Chloride_max,Chloride_std,Chloride_last,Creatinine_mean,Creatinine_min,Creatinine_max,Creatinine_std,Creatinine_last,Bilirubin_direct_mean,Bilirubin_direct_min,Bilirubin_direct_max,Bilirubin_direct_std,Bilirubin_direct_last,Glucose_mean,Glucose_min,Glucose_max,Glucose_std,Glucose_last,Lactate_mean,Lactate_min,Lactate_max,Lactate_std,Lactate_last,Magnesium_mean,Magnesium_min,Magnesium_max,Magnesium_std,Magnesium_last,Phosphate_mean,Phosphate_min,Phosphate_max,Phosphate_std,Phosphate_last,Potassium_mean,Potassium_min,Potassium_max,Potassium_std,Potassium_last,Bilirubin_total_mean,Bilirubin_total_min,Bilirubin_total_max,Bilirubin_total_std,Bilirubin_total_last,TroponinI_mean,TroponinI_min,TroponinI_max,TroponinI_std,TroponinI_last,Hct_mean,Hct_min,Hct_max,Hct_std,Hct_last,Hgb_mean,Hgb_min,Hgb_max,Hgb_std,Hgb_last,PTT_mean,PTT_min,PTT_max,PTT_std,PTT_last,WBC_mean,WBC_min,WBC_max,WBC_std,WBC_last,Fibrinogen_mean,Fibrinogen_min,Fibrinogen_max,Fibrinogen_std,Fibrinogen_last,Platelets_mean,Platelets_min,Platelets_max,Platelets_std,Platelets_last,Age_mean,Age_min,Age_max,Age_std,Age_last,Gender_mean,Gender_min,Gender_max,Gender_std,Gender_last,Unit1_mean,Unit1_min,Unit1_max,Unit1_std,Unit1_last,Unit2_mean,Unit2_min,Unit2_max,Unit2_std,Unit2_last,HospAdmTime_mean,HospAdmTime_min,HospAdmTime_max,HospAdmTime_std,HospAdmTime_last,ICULOS_mean,ICULOS_min,ICULOS_max,ICULOS_std,ICULOS_last,SepsisLabel
0,101.571429,76.0,117.0,9.594378,84.0,91.477273,85.0,100.0,3.460667,85.0,36.778000,36.11,37.44,0.421078,NaN,126.809524,78.0,181.0,22.422482,78.0,87.261905,44.00,141.33,21.270465,44.0,NaN,NaN,NaN,NaN,NaN,24.820000,17.0,32.0,4.106689,18.0,NaN,NaN,NaN,NaN,NaN,20.714286,18.0,24.0,2.13809,NaN,46.500000,45.0,48.0,2.121320,NaN,0.282500,0.25,0.3,0.023629,NaN,7.347143,7.31,7.40,0.031472,NaN,95.333333,86.0,100.0,6.022181,NaN,86.500000,78.0,91.0,5.802298,NaN,16.000000,16.0,16.0,NaN,NaN,18.000000,14.0,22.0,5.656854,NaN,98.0,98.0,98.0,NaN,NaN,9.450000,9.3,9.6,0.212132,NaN,85.000000,85.0,85.0,0.00000,NaN,0.700000,0.7,0.7,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,163.000000,133.0,193.0,42.426407,NaN,NaN,NaN,NaN,NaN,NaN,2.100000,2.0,2.2,0.141421,NaN,3.500000,3.3,3.7,0.282843,NaN,4.200000,3.8,4.6,0.565685,NaN,0.300000,0.3,0.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,36.700000,36.2,37.2,0.707107,NaN,12.350000,12.2,12.5,0.212132,NaN,NaN,NaN,NaN,NaN,NaN,10.200000,5.7,14.7,6.363961,NaN,NaN,NaN,NaN,NaN,NaN,327.500000,317.0,338.0,14.849242,NaN,83.14,83.14,83.14,2.868859e-14,83.14,0.0,0,0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.03,-0.03,-0.03,3.502025e-18,-0.03,27.5,1,54,15.732133,54,0
1,60.954545,54.0,94.0,8.144395,55.0,97.000000,94.0,100.0,2.138090,95.0,36.165000,36.00,36.44,0.151625,NaN,136.600000,114.0,194.0,20.180613,NaN,66.704545,50.50,116.00,14.720134,51.0,44.066667,36.0,66.0,7.095941,NaN,14.236842,9.0,27.0,4.667763,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.000000,22.0,22.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.000000,100.0,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.900000,7.9,7.9,NaN,NaN,113

In [11]:
missing = (
    external_feature_dataset
    .isnull()
    .mean()
    .sort_values(ascending=False) * 100
)

missing.head(20)

Bilirubin_direct_last    99.888437
Fibrinogen_last          99.672749
TroponinI_last           99.543832
Bilirubin_total_last     99.253768
AST_last                 99.201706
Alkalinephos_last        99.199226
Lactate_last             98.948830
Bilirubin_direct_std     98.787683
SaO2_last                98.537287
PTT_last                 98.358786
EtCO2_last               98.019139
BaseExcess_last          97.657180
Phosphate_last           97.634867
PaCO2_last               97.619992
HCO3_last                97.533221
Chloride_last            97.347283
pH_last                  96.806823
Calcium_last             96.782031
Platelets_last           96.715093
Creatinine_last          96.501884
dtype: float64

In [12]:
output_path = Path("../data/processed")
output_path.mkdir(exist_ok=True)

external_feature_dataset.to_csv(
    output_path / "external_feature_dataset.csv",
    index=False
)

print("External feature dataset saved successfully!")

External feature dataset saved successfully!
